# Model 1 — Inference Only
**Team11 | WU LLM Course SS26**

Runs a local Mistral-7B-Instruct model on all 643 Austrian tax law questions.
No retrieval, no fine-tuning — pure prompt-based inference.

**Platform:** Google Colab free T4 GPU  
**Runtime:** ~20–40 min for 643 questions on T4

## Cell 1 — Install Libraries & Imports

Installs all required packages (`transformers`, `accelerate`, `bitsandbytes`, `torch`, `pandas`) and imports them at the top of the notebook.

**Impact:** Placing all imports in one cell prevents `NameError` after a Colab kernel restart — every later cell depends on this running first.

In [ ]:
# Cell 1 — Install libraries
!pip install -q transformers accelerate bitsandbytes torch pandas

# imports — all here so a kernel restart doesn't break later cells
import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

## Cell 2 — Upload Dataset

Opens a file picker so you can upload `dataset_clean.csv` from your local machine into the Colab runtime. The file contains 643 Austrian tax law questions.

**Impact:** Without this file the inference loop has no questions to process — this cell must run before any generation step.

In [ ]:
# Cell 2 — Upload dataset_clean.csv
# Run this cell, click "Choose Files", select dataset_clean.csv from your machine
from google.colab import files
uploaded = files.upload()  # upload dataset_clean.csv

## Cell 3 — Load Model in 4-bit Quantization

Downloads `Mistral-7B-Instruct-v0.2` and loads it in 4-bit NF4 quantization via `BitsAndBytesConfig`. The `device_map="auto"` flag automatically places layers on the available GPU.

**Impact:** 4-bit quantization reduces memory from ~14 GB to ~4 GB, making a 7-billion-parameter model fit on the free Colab T4 GPU (16 GB VRAM). Without this step the model would not load.

In [ ]:
# Cell 3 — Load model + tokenizer in 4-bit quantization
# 4-bit keeps memory usage low enough to fit on the free T4 GPU
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # freely accessible, no HF token needed

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"  # automatically uses the GPU
)

print("Model loaded.")

## Cell 4 — Load Dataset & Define System Prompt

Reads the uploaded CSV into a DataFrame and defines the shared system prompt used across all three models. The `build_prompt()` helper formats each question as a Mistral-style chat message.

**Impact:** The system prompt is identical across Models 1, 2, and 3 — this controls for prompt differences so any quality gap reflects the model architecture, not the instructions.

In [ ]:
# Cell 4 — Load dataset and define shared prompt
import pandas as pd
import os

df = pd.read_csv("dataset_clean.csv")
print(f"Loaded {len(df)} questions.")

# Shared system prompt — same across all 3 models for fair comparison
# Citation format: § 26 Abs 1 Z 2 lit a EStG 1988  (no dot after Abs)
SYSTEM_PROMPT = (
    "Beantworte die folgende Frage zum österreichischen Steuerrecht auf Deutsch. "
    "Antworte in maximal 1–4 Sätzen. "
    "Nenne die einschlägige Rechtsnorm (z.B. § 7 Abs 1 KStG). "
    "Halluziniere keine Paragraphen. Wenn unklar, formuliere vorsichtig."
)

def build_prompt(question):
    # Format as Mistral instruction turn
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + question}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## Cell 5 — Main Inference Loop

Iterates over all 643 questions and generates an answer for each using greedy decoding (`do_sample=False`, `max_new_tokens=200`). A checkpoint CSV is saved every 50 rows so progress survives a session crash.

**Impact:** This is the core cell that produces all answers. Greedy decoding makes outputs deterministic and reproducible. The checkpoint lets you resume from where you left off instead of restarting from scratch.

In [ ]:
# Cell 5 — Generate answers with checkpointing every 50 rows
# If the session crashes, re-run from this cell — it will resume from the checkpoint
CHECKPOINT = "checkpoint_model1.csv"
SAVE_EVERY = 50

# load checkpoint if it exists (resume after crash)
if os.path.exists(CHECKPOINT):
    done = pd.read_csv(CHECKPOINT)
    results = done.to_dict("records")
    done_ids = set(done["id"])
    print(f"Resuming from checkpoint: {len(results)} already done.")
else:
    results = []
    done_ids = set()

# loop through all questions
for i, row in df.iterrows():
    if row["id"] in done_ids:
        continue  # skip already completed

    prompt = build_prompt(row["prompt"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # generate answer — keep it short (max 200 new tokens)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,        # greedy — deterministic output
            pad_token_id=tokenizer.eos_token_id
        )

    # decode only the newly generated tokens (skip the prompt)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    results.append({"id": row["id"], "answer": answer})

    # save checkpoint every 50 rows
    if len(results) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)
        print(f"  Checkpoint saved at row {len(results)}")

print(f"Done. {len(results)} answers generated.")

## Cell 6 — Save Results to CSV

Writes all generated answers to `model1_inference.csv` and triggers a download to your local machine.

**Impact:** Produces the final submission file in the required `id, answer` format. Download it immediately — Colab runtimes are temporary and files are lost when the session ends.

In [ ]:
# Cell 6 — Save final results to CSV
output_path = "model1_inference.csv"
pd.DataFrame(results).to_csv(output_path, index=False)
print(f"Saved to {output_path}")

# download the file to your machine
files.download(output_path)

## Cell 7 — Validate Submission Format

Checks the output CSV for common formatting errors: wrong column names, wrong row count, mismatched IDs, null or blank answers, and repeated answers (which would indicate a generation crash).

**Impact:** Catches submission errors before the file is copied to `results/`. A formatting mistake here would silently break automated evaluation and cost points.

In [ ]:
# Cell 7 — Validate submission format
# Checks the output CSV before you move it to Team11/results/
ref = pd.read_csv("dataset_clean.csv")
sub = pd.read_csv(output_path)

assert list(sub.columns) == ["id", "answer"], "Wrong columns"
assert len(sub) == len(ref), f"Expected {len(ref)} rows, got {len(sub)}"
assert sub["id"].tolist() == ref["id"].tolist(), "IDs don't match"
assert sub["id"].nunique() == len(sub), "Duplicate IDs"
assert sub["answer"].notna().all(), "Null answers found"
assert (sub["answer"].str.strip() != "").all(), "Blank answers found"
top_freq = sub["answer"].value_counts().iloc[0]
assert top_freq < 20, f"Repeated answer {top_freq}x — possible crash"

print(f"OK — {output_path} is valid ({len(sub)} rows)")
print("\nSample answers:")
print(sub.head(3).to_string())